# 💧 LFM2 Inference with llama.cpp

This notebook demonstrates how to run [LFM2](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda) and [LFM2.5](https://huggingface.co/collections/LiquidAI/lfm25-6839e3e26b2a9fdbde95b341) models using [llama.cpp](https://github.com/ggml-org/llama.cpp) for efficient CPU inference.

No GPU required - runs on CPU!

## Installation

Download and extract the llama.cpp binaries:

In [ ]:
!wget https://github.com/ggml-org/llama.cpp/releases/download/b7633/llama-b7633-bin-ubuntu-x64.tar.gz
!tar -xzf llama-b7633-bin-ubuntu-x64.tar.gz

## Basic Text Generation

Run inference using `llama-cli` with the `-hf` flag to download models directly from Hugging Face:

In [ ]:
# !modal_skip
!llama-b7633/llama-cli \
    -hf LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M \
    -p "What is C. elegans?" \
    -n 256 \
    --temp 0.1 --top-k 50 --top-p 0.1 --repeat-penalty 1.05

## Using llama-server with OpenAI API

Start the server and use the OpenAI Python client. We use `subprocess` here to start the server within the notebook — you can also run the server command directly in a terminal.

In [ ]:
import subprocess
import time

server = subprocess.Popen(
    ["llama-b7633/llama-server",
     "-hf", "LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M",
     "-c", "4096",
     "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(30)
print("Server running!")

Check server health — if it doesn't print "ok", the model is still loading:

In [ ]:
!curl -s http://localhost:8000/health

In [ ]:
!uv pip install -qqq openai requests

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="lfm2.5-1.2b-instruct",
    messages=[
        {"role": "user", "content": "What is machine learning?"}
    ],
    temperature=0.1,
    top_p=0.1,
    max_tokens=512,
    extra_body={"top_k": 50, "repetition_penalty": 1.05},
)
print(response.choices[0].message.content)

In [ ]:
# Stop the server
server.terminate()

## Vision Models

LFM2-VL models support image inputs:

In [ ]:
import requests

# Download a test image
image_url = "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
img_data = requests.get(image_url).content
with open("test_image.jpg", "wb") as f:
    f.write(img_data)

In [ ]:
# !modal_skip
!llama-b7633/llama-cli \
    -hf LiquidAI/LFM2.5-VL-1.6B-GGUF:Q4_0 \
    --image test_image.jpg \
    --image-max-tokens 64 \
    -p "What's in this image?" \
    -n 128 \
    --temp 0.1 --min-p 0.15 --repeat-penalty 1.05

### Using llama-server with OpenAI API

Start the vision model server.

In [ ]:
import subprocess
import time

server = subprocess.Popen(
    ["llama-b7633/llama-server",
     "-hf", "LiquidAI/LFM2.5-VL-1.6B-GGUF:Q4_0",
     "-c", "4096",
     "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(30)
print("Server running!")

Check server health — if it doesn't print "ok", the model is still loading:

In [ ]:
!curl -s http://localhost:8000/health

In [ ]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="lfm2.5-vl-1.6b",
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": "What's in this image?"}
        ]
    }],
    temperature=0.1,
    max_tokens=512,
    extra_body={"min_p": 0.15, "repetition_penalty": 1.05},
)
print(response.choices[0].message.content)

In [ ]:
# Stop the server
server.terminate()

## Resources

- [LFM2 Documentation](https://docs.liquid.ai/docs/inference/llama-cpp)
- [LFM2 GGUF Models on Hugging Face](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF)
- [llama.cpp Documentation](https://github.com/ggml-org/llama.cpp)